In [76]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.6 MB/s eta 0:00:00


In [268]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.compose import ColumnTransformer

from skopt import BayesSearchCV

import statsmodels.api as sm
import scipy.stats as stats

In [21]:
df = pd.read_csv('https://raw.githubusercontent.com/ichiP245/TP_Kaggle_Predictivo/refs/heads/main/df_train_0_noOutliers.csv')

In [22]:
def root_mean_squared_error(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

In [23]:
dummies_genre = pd.get_dummies(df['playlist_genre']).astype(int)
dummies_subgenre = pd.get_dummies(df['playlist_subgenre']).astype(int)
dummies_grupo_anio = pd.get_dummies(df['grupo_anio']).astype(int)
variables_RL = ['danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'duration_ms']

df_RL = pd.concat([df[variables_RL], dummies_genre, dummies_subgenre, dummies_grupo_anio], axis=1)

X = df_RL.copy()
y = df['track_popularity'].copy()

### Regresión Lineal Multiple

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
# Hacemos modelo con StatsModels -> como en las dummies nunca eliminamos una categoria, ahora no agregamos intercepto
# Fit the linear regression model
model = sm.OLS(y_train, X_train).fit()

# Generate a summary of the regression results
summary = model.summary()

print(summary)

                            OLS Regression Results                            
Dep. Variable:       track_popularity   R-squared:                       0.180
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     112.0
Date:                Fri, 22 May 2026   Prob (F-statistic):               0.00
Time:                        12:35:29   Log-Likelihood:                -95331.
No. Observations:               21012   AIC:                         1.907e+05
Df Residuals:                   20970   BIC:                         1.911e+05
Df Model:                          41                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
danceability           8.3606      1

Variables no significativas: key, mode, speechiness, acousticness, duration_ms y subgenre 'new jack swing'

In [26]:
y_pred = model.predict(X_test)

print(f'Middle Square Error = {mean_squared_error(y_test, y_pred):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred):.2f}')

Middle Square Error = 513.61
Root Middle Square Error = 22.66
R2 = 0.19


Hacemos tambien feature_selection

In [27]:
def forward_selection(X, y,AIC=True, verbose=True):
    remaining = list(X.columns)
    selected = []
    current_score, best_new_score = np.inf, np.inf
    while remaining:
        scores_with_candidates = []
        for candidate in remaining:
            formula = selected + [candidate]
            X_model = sm.add_constant(X[formula])
            model = sm.OLS(y, X_model).fit()
            if AIC:
              aic = model.aic
              scores_with_candidates.append((aic, candidate))
            else:
              bic = model.bic
              scores_with_candidates.append((bic, candidate))
        scores_with_candidates.sort()
        best_new_score, best_candidate = scores_with_candidates[0]
        if current_score > best_new_score:
            remaining.remove(best_candidate)
            selected.append(best_candidate)
            current_score = best_new_score
            if verbose:
              if AIC:
                print(f"Añadido: {best_candidate}, AIC = {current_score:.2f}")
              else:
                print(f"Añadido: {best_candidate}, BIC = {current_score:.2f}")
        else:
            break
    return selected

In [29]:
seleccionadas = forward_selection(X, y)

Añadido: 2019, AIC = 242221.82
Añadido: new jack swing, AIC = 241868.39
Añadido: 2000, AIC = 241576.94
Añadido: post-teen pop, AIC = 241272.27
Añadido: permanent wave, AIC = 240979.13
Añadido: 2010, AIC = 240709.00
Añadido: instrumentalness, AIC = 240485.75
Añadido: gangster rap, AIC = 240319.79
Añadido: rock, AIC = 240124.09
Añadido: neo soul, AIC = 239972.85
Añadido: southern hip hop, AIC = 239808.96
Añadido: energy, AIC = 239688.82
Añadido: loudness, AIC = 239187.16
Añadido: 2015, AIC = 239070.31
Añadido: 2016, AIC = 238948.35
Añadido: 2017, AIC = 238871.93
Añadido: tropical, AIC = 238807.98
Añadido: indie poptimism, AIC = 238734.34
Añadido: danceability, AIC = 238685.97
Añadido: 1970, AIC = 238637.11
Añadido: 1980, AIC = 238586.45
Añadido: latin hip hop, AIC = 238537.72
Añadido: trap, AIC = 238494.54
Añadido: electropop, AIC = 238470.48
Añadido: hard rock, AIC = 238460.54
Añadido: urban contemporary, AIC = 238454.12
Añadido: latin pop, AIC = 238448.71
Añadido: dance pop, AIC = 2384

In [33]:
print("\nCantidad de variables seleccionadas: ",len(seleccionadas))
print("\nCantidad de variables totales: ",len(X.columns))
print("\n Variables no seleccionadas: ", (set(X.columns) - set(seleccionadas)))


Cantidad de variables seleccionadas:  34

Cantidad de variables totales:  48

 Variables no seleccionadas:  {'speechiness', 2018, 1990, 'classic rock', 'hip pop', 'mode', 'r&b', 'hip hop', 'rap', 'key', 'duration_ms', 'reggaeton', 'pop', 'latin'}


### Regresion Polinomial: simple + lasso

In [44]:
X_train_poly = X_train[['danceability','energy','loudness','acousticness','instrumentalness','liveness','valence','tempo','duration_ms']]
X_test_poly = X_test[['danceability','energy','loudness','acousticness','instrumentalness','liveness','valence','tempo','duration_ms']]

poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly_g3 = poly.fit_transform(X_train_poly)
X_test_poly_g3 = poly.transform(X_test_poly)

# Entrenamos modelo lineal grado 3
print('Modelo lineal polinomial grado 3')
reg_linear = LinearRegression()
reg_linear.fit(X_train_poly_g3, y_train)
y_pred_poly = reg_linear.predict(X_test_poly_g3)

print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_poly):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_poly):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_poly):.2f}')

# Entrenamos modelo regularizado grado 3
print('Modelo Lasso polinomial grado 3')
reg_lasso = Lasso(alpha=0.1)
reg_lasso.fit(X_train_poly_g3, y_train)
y_pred_poly_lasso = reg_lasso.predict(X_test_poly_g3)

print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_poly_lasso):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_poly_lasso):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_poly_lasso):.2f}')

Modelo lineal polinomial grado 3
Middle Square Error = 597.98
Root Middle Square Error = 24.45
R2 = 0.05
Modelo Lasso polinomial grado 3
Middle Square Error = 592.00
Root Middle Square Error = 24.33
R2 = 0.06


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.036e+06, tolerance: 1.308e+03
  model = cd_fast.enet_coordinate_descent(


In [45]:
### Probamos modelo regularizado grado 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly_g2 = poly.fit_transform(X_train_poly)
X_test_poly_g2 = poly.transform(X_test_poly)

print('Modelo Lasso polinomial grado 2')
reg_lasso = Lasso(alpha=0.1, max_iter=2500)
reg_lasso.fit(X_train_poly, y_train)
y_pred_poly_lasso = reg_lasso.predict(X_test_poly)

print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_poly_lasso):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_poly_lasso):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_poly_lasso):.2f}')

Modelo Lasso polinomial grado 2
Middle Square Error = 598.07
Root Middle Square Error = 24.46
R2 = 0.05


### Regresion Ridge optimizada

In [56]:
degree = 1
alpha = 0.01
pipeline = Pipeline([
    # ("poly", PolynomialFeatures(degree=degree)),  # Vamos a querer que le aplique esto a cada conjunto de X
    ("scaler", StandardScaler()),   # Vamos a querer que estandarize (para mejor funcionamiento de la regularizacion)
    ("ridge", Ridge(alpha=alpha, max_iter=10000)) # Vamos a querer que aplique la regresion con regularizacion
])

param_grid = {"ridge__alpha": np.logspace(-3, 1, 40),
              # "poly__degree": np.arange(1,3)  # tambien podría agregar distintos valores de "degree" a la búsqueda
}

grid_search = GridSearchCV(
    pipeline,   # Definido antes
    param_grid, # Con aquello que queres optimizar. Puede ser solo el alpha o solo el grado
    cv=5,       # Cantidad de particiones/splits
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)
grid_search.fit(X_train, y_train) # Usamos el X_train e y_train del principio

print("Mejores hiperparametros:", grid_search.best_params_)
print("Mejor RMSE (CV):", -grid_search.best_score_)

Mejores hiperparametros: {'ridge__alpha': np.float64(10.0)}
Mejor RMSE (CV): 22.65794081052057


In [63]:
y_pred_best_ridge = grid_search.best_estimator_.predict(X_test)

print('Modelo Ridge alpha optimo')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_best_ridge):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_best_ridge):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_best_ridge):.2f}')

Modelo Ridge alpha optimo
Middle Square Error = 513.62
Root Middle Square Error = 22.66
R2 = 0.19


### SVM regressor

Mejor R^2 de 0,19
Mejor RMSE de 22,59

In [59]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [62]:
svr = SVR() # kernel = 'rbf', C=1, epsilon=0.1
svr.fit(X_train_scaled, y_train)

SVR()

In [64]:
y_pred_svm = svr.predict(X_test_scaled)

print('Modelo SVR basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_svm):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_svm):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_svm):.2f}')

Modelo SVR basico
Middle Square Error = 510.29
Root Middle Square Error = 22.59
R2 = 0.19


Tratamos de optimizarlo

In [69]:
features_ok = list(set(X_train.columns) - set(['key', 'mode', 'speechiness', 'acousticness', 'duration_ms', 'new jack swing','speechiness', '2018', '1990', 'classic rock', 'hip pop', 'mode', 'hip hop', 'rap', 'key', 'duration_ms', 'reggaeton', 'pop', 'latin']))

In [73]:
# Sacamos subgenre -> pero dejo rock que era significativa y los de r&b y neosoul que aparecen en val
features_final = ['rock', 'r&b', 'neo soul', 'instrumentalness','liveness',  'loudness', 'tempo', 'valence','energy', 'danceability',
       '1980', '1970', '2000', '2010','2015', '2016', '2017', '2019',  '2020']

In [87]:
X_train_SVR = X_train[features_final].copy()
X_test_SVR = X_test[features_final].copy()

scaler = StandardScaler()
X_train_SVR_scaled = scaler.fit_transform(X_train_SVR)
X_test_SVR_scaled = scaler.transform(X_test_SVR)

Probamos otros SVR

In [88]:
# kernel = 'rbf', C=10, epsilon = 0.1 (mantenemos)
svr = SVR(kernel='rbf', C=10, epsilon=0.1)
svr.fit(X_train_SVR_scaled, y_train)

y_pred_svm = svr.predict(X_test_SVR_scaled)

print('Modelo SVR basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_svm):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_svm):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_svm):.2f}')

Modelo SVR basico
Middle Square Error = 555.68
Root Middle Square Error = 23.57
R2 = 0.12


In [96]:
# kernel = 'rbf', C=1, epsilon = 0.5
svr = SVR(kernel='rbf', C=0.5, epsilon=0.05)
svr.fit(X_train_SVR_scaled, y_train)

y_pred_svm = svr.predict(X_test_SVR_scaled)

print('Modelo SVR basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_svm):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_svm):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_svm):.2f}')

Modelo SVR basico
Middle Square Error = 560.17
Root Middle Square Error = 23.67
R2 = 0.11


In [89]:
# kernel = 'rbf', C=0.5, epsilon = 0.8 -> buscamos que con menor C regularize mas y con mayor epsilon penalice mas
svr = SVR(kernel='rbf', C=0.5, epsilon=0.8)
svr.fit(X_train_SVR_scaled, y_train)

y_pred_svm = svr.predict(X_test_SVR_scaled)

print('Modelo SVR basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_svm):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_svm):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_svm):.2f}')

Modelo SVR basico
Middle Square Error = 560.18
Root Middle Square Error = 23.67
R2 = 0.11


In [93]:
svr = SVR(kernel='rbf', C=0.01, epsilon=5)
svr.fit(X_train_SVR_scaled, y_train)

y_pred_svm = svr.predict(X_test_SVR_scaled)

print('Modelo SVR basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_svm):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_svm):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_svm):.2f}')

Modelo SVR basico
Middle Square Error = 622.60
Root Middle Square Error = 24.95
R2 = 0.02


In [90]:
# kernel = 'linear' (cambiamos), C=1 (cambiamos), epsilon = 0.1 (mantenemos)
svr = SVR(kernel='linear', C=1, epsilon=0.1)
svr.fit(X_train_SVR_scaled, y_train)

y_pred_svm = svr.predict(X_test_SVR_scaled)

print('Modelo SVR basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_svm):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_svm):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_svm):.2f}')

Modelo SVR basico
Middle Square Error = 570.97
Root Middle Square Error = 23.90
R2 = 0.10


### KNN regressor

In [101]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [102]:
# Hago un KNN normal -> con todas las variables
'''
n_neighbors=5, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski'
'''
knnr = KNeighborsRegressor()
knnr.fit(X_train_scaled, y_train)

y_pred_knn = knnr.predict(X_test_scaled)

print('Modelo KNN basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_knn):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_knn):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_knn):.2f}')

Modelo KNN basico
Middle Square Error = 574.19
Root Middle Square Error = 23.96
R2 = 0.09


Probamos con menos variables

In [103]:
# Hago un KNN normal -> con menos variables
'''
n_neighbors=5, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski'
'''

X_train_KNN = X_train[features_final].copy()
X_test_KNN = X_test[features_final].copy()

scaler = StandardScaler()
X_train_KNN_scaled = scaler.fit_transform(X_train_KNN)
X_test_KNN_scaled = scaler.transform(X_test_KNN)

knnr = KNeighborsRegressor()
knnr.fit(X_train_KNN_scaled, y_train)

y_pred_knn = knnr.predict(X_test_KNN_scaled)

print('Modelo KNN basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_knn):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_knn):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_knn):.2f}')

Modelo KNN basico
Middle Square Error = 604.25
Root Middle Square Error = 24.58
R2 = 0.04


Vamos con todas las variables otra vez -> probamos otras configuraciones

In [219]:
# Hago un KNN normal -> con menos variables
'''
n_neighbors=100, *, weights='distance', algorithm='auto', leaf_size=20, p=2, metric='manhattan'
'''
knnr = KNeighborsRegressor(n_neighbors=100, weights='distance', metric='manhattan', leaf_size=20) # -> parametro 'distance' fundamental, uso de k ~raiz cuadrada de sample
knnr.fit(X_train_KNN_scaled, y_train)

y_pred_knn = knnr.predict(X_test_KNN_scaled)

print('Modelo KNN basico')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_knn):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_knn):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_knn):.2f}')

Modelo KNN basico
Middle Square Error = 470.61
Root Middle Square Error = 21.69
R2 = 0.26


Corro con cross validation para verificar

In [246]:
kF = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(knnr, X_train_KNN_scaled, y_train, cv=kF, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

RMSE por fold: [21.59258494 22.22766508 22.25355038 22.14657382 22.15767326]
RMSE promedio: 22.075609493377396
Desvío estándar del RMSE: 0.24489466271831653


Agrego las variables numericas que habia descartado y vuelvo a correr

In [245]:
# Hago un KNN normal -> con menos variables
'''
n_neighbors=5, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski'
'''

X_train_KNN_plus = X_train[features_final+['mode', 'speechiness', 'acousticness', 'duration_ms']].copy()
X_test_KNN_plus = X_test[features_final+['mode', 'speechiness', 'acousticness', 'duration_ms']].copy()

numeric_cols = X_train_KNN_plus.select_dtypes(include='float')
dummy_cols = X_train_KNN_plus.select_dtypes(include='int')

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols.columns.tolist()),
    ("cat", "passthrough", dummy_cols.columns.tolist())
])

model = Pipeline([
    ("prep", preprocessor),
    ("knn", KNeighborsRegressor(n_neighbors=100, weights='distance', metric='manhattan', leaf_size=20))
])

scores_knn = cross_val_score(model, X_train_KNN_plus, y_train, cv=kF, scoring='neg_root_mean_squared_error')
print('Modelo KNN plus - Cross Val')
print("RMSE por fold:", -scores_knn)
print("RMSE promedio:", -np.mean(scores_knn))
print("Desvío estándar del RMSE:", np.std(-scores_knn))
model.fit(X_train_KNN_plus, y_train)

y_pred_knn_plus = model.predict(X_test_KNN_plus)
print('Modelo KNN plus - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_knn_plus):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_knn_plus):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_knn_plus):.2f}')

Modelo KNN plus - Cross Val
RMSE por fold: [21.71565862 22.35012246 22.38227602 22.21262745 22.20822642]
RMSE promedio: 22.173782194183993
Desvío estándar del RMSE: 0.23964061785913618
Modelo KNN plus - Test
Middle Square Error = 474.69
Root Middle Square Error = 21.79
R2 = 0.25


#### Hago una submission

Hago prediccon de Val con este modelo

In [200]:
df_val = pd.read_csv('https://raw.githubusercontent.com/ichiP245/TP_Kaggle_Predictivo/refs/heads/main/base_val.csv')

In [202]:
def imputacion_anio(anio: int):
  if anio < 1980:
    return 1970
  if anio < 1990:
    return 1980
  if anio < 2000:
    return 1990
  if anio < 2010:
    return 2000
  if anio < 2015:
    return 2010
  return anio

In [201]:
df_val['anio'] = df_val['track_album_release_date'].apply(lambda x: x.split('-')[0])
df_val['anio'] = df_val['anio'].astype(int)

In [203]:
df_val['grupo_anio'] = df_val['anio'].apply(imputacion_anio)

In [204]:
df_val_orig = df_val.copy()

In [205]:
dummies_genre = pd.get_dummies(df_val['playlist_genre']).astype(int)
dummies_subgenre = pd.get_dummies(df_val['playlist_subgenre']).astype(int)
dummies_grupo_anio = pd.get_dummies(df_val['grupo_anio']).astype(int)
variables = ['danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'duration_ms']

df_val = pd.concat([df_val[variables], dummies_genre, dummies_subgenre, dummies_grupo_anio], axis=1)

In [206]:
df_val.columns = df_val.columns.astype(str)
df_val.columns = df_val.columns.astype(str)

In [207]:
df_val['rock'] = 0

In [208]:
len(X_train_KNN.columns)

19

In [209]:
df_val_knn = df_val[list((set(df_val.columns) & set(X_train_KNN.columns)))]
df_val_knn_scaled = scaler.transform(df_val_knn[features_final].copy())

In [210]:
y_pred_val_knn = knnr.predict(df_val_knn_scaled)

In [212]:
df_val_orig['Unnamed: 0']

,Unnamed: 0
0,26266
1,26267
2,26268
3,26269
4,26270
...,...
6562,32828
6563,32829
6564,32830
6565,32831


In [214]:
submission_df_cb = pd.DataFrame({
    'ID': df_val_orig['Unnamed: 0'],
    'track_popularity': y_pred_val_knn
})

submission_df_cb.to_csv('submission_knn.csv', index=False)

### Random Forest regressor

Corremos un RF Regressor sin restricciones para ver las features mas importantes

In [251]:
rfr = RandomForestRegressor(n_jobs=-1, random_state=42)
rfr.fit(X_train, y_train)
scores = cross_val_score(rfr, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

RMSE por fold: [21.46215917 21.42633092 21.49877598 21.14139798 21.27335012]
RMSE promedio: 21.36040283485498
Desvío estándar del RMSE: 0.1337081091128502


In [252]:
y_pred_rfr_base = rfr.predict(X_test)
print('Modelo RFR base - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_rfr_base):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_rfr_base):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_rfr_base):.2f}')

Modelo RFR base - Test
Middle Square Error = 446.63
Root Middle Square Error = 21.13
R2 = 0.29


In [253]:
feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rfr.feature_importances_
})
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)
feature_importances

,Feature,Importance
3,loudness,0.086114
11,duration_ms,0.076090
1,energy,0.075382
10,tempo,0.073175
5,speechiness,0.072675
6,acousticness,0.072237
9,valence,0.072217
0,danceability,0.071988
8,liveness,0.069056
7,instrumentalness,0.061243


In [258]:
feature_importances.head(20)

,Feature,Importance
3,loudness,0.086114
11,duration_ms,0.076090
1,energy,0.075382
10,tempo,0.073175
5,speechiness,0.072675
6,acousticness,0.072237
9,valence,0.072217
0,danceability,0.071988
8,liveness,0.069056
7,instrumentalness,0.061243


Corremos otra vez, pero solo con las features que importan para validation set

In [265]:
vars = ['2019', 'loudness', 'energy', 'instrumentalness', 'duration_ms',
       'danceability', 'acousticness', 'tempo', 'speechiness', 'valence',
       'liveness', 'key','2018', '2000','mode','neo soul','r&b','2017', '2015', '1970', '2016', '1990','2020','1980']

In [274]:
X_train_reduced = X_train[vars]
X_test_reduced = X_test[vars]

rfr = RandomForestRegressor(n_jobs=-1, random_state=42)
rfr.fit(X_train_reduced, y_train)
scores = cross_val_score(rfr, X_train_reduced, y_train, cv=5, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

y_pred_rfr_base = rfr.predict(X_test_reduced)
print('Modelo RFR base - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_rfr_base):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_rfr_base):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_rfr_base):.2f}')

RandomForestRegressor(n_jobs=-1, random_state=42)

Hacemos uno en particular

In [231]:
# n_estimators=100, *, criterion='squared_error', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0,
# max_features=1.0, max_leaf_nodes=None, min_impurity_decrease=0.0,
# bootstrap=True, oob_score=False, n_jobs=None, random_state=None
rfr = RandomForestRegressor(max_depth=10, min_samples_split=20, max_features='sqrt', oob_score=True, n_jobs=-1, random_state=42)
rfr.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, max_features='sqrt', min_samples_split=20,
                      n_jobs=-1, oob_score=True, random_state=42)

In [233]:
cross_val_score(rfr, X_train, y_train, cv=5)

array([0.18201029, 0.17890666, 0.1799603 , 0.18997583, 0.18907784])

In [235]:
len(X_train.columns)

48

In [232]:
y_pred_rfr = rfr.predict(X_test)

print('Modelo RFR - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_rfr):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_rfr):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_rfr):.2f}')

Modelo RFR - Test
Middle Square Error = 509.44
Root Middle Square Error = 22.57
R2 = 0.19


Pruebo otros modelos

In [269]:
rfr = RandomForestRegressor(max_depth=30, min_samples_split=100, max_features=0.8, n_jobs=-1, random_state=42)
rfr.fit(X_train_reduced, y_train)
scores = cross_val_score(rfr, X_train_reduced, y_train, cv=5, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

RMSE por fold: [22.60150796 22.6513531  22.69689724 22.45314848 22.6171117 ]
RMSE promedio: 22.604003696609315
Desvío estándar del RMSE: 0.08222715291695808


In [270]:
r2_score(y_test, rfr.predict(X_test_reduced))

0.20254172237493617

Intento optimizar

In [248]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.4 MB/s eta 0:00:00


Con diccionario de parametros extenso

In [249]:
import optuna

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),

        "max_depth": trial.suggest_int("max_depth", 5, 30),

        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),

        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),

        "max_features": trial.suggest_float("max_features", 0.3, 1.0),

        "n_jobs": -1,
        "random_state": 42
    }

    model = RandomForestRegressor(**params)

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=25
)

print(study.best_params)
print(-study.best_value)

[I 2026-05-22 16:15:15,776] A new study created in memory with name: no-name-cb686c92-c4e9-41c8-915c-d067009b96df
[I 2026-05-22 16:19:39,018] Trial 0 finished with value: -21.589605582522683 and parameters: {'n_estimators': 485, 'max_depth': 28, 'min_samples_split': 9, 'min_samples_leaf': 6, 'max_features': 0.8007391950061158}. Best is trial 0 with value: -21.589605582522683.
[I 2026-05-22 16:20:54,350] Trial 1 finished with value: -22.366346021784306 and parameters: {'n_estimators': 290, 'max_depth': 9, 'min_samples_split': 15, 'min_samples_leaf': 9, 'max_features': 0.7151612037964864}. Best is trial 0 with value: -21.589605582522683.
[I 2026-05-22 16:22:45,633] Trial 2 finished with value: -22.245366589781582 and parameters: {'n_estimators': 293, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 8, 'max_features': 0.9767815743531147}. Best is trial 0 with value: -21.589605582522683.
[I 2026-05-22 16:25:25,418] Trial 3 finished with value: -21.547870893956766 and parameters

KeyboardInterrupt: 

Lo interrumpi en el trial 15. Estaba asi:

**[I 2026-05-22 16:42:29,008] Trial 11 finished with value: -21.53529043415121 and parameters: {'n_estimators': 203, 'max_depth': 22, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 0.8991086774270771}. Best is trial 11 with value: -21.53529043415121.**

Con diccionario de parametros mas acotado

In [250]:
import optuna

def objective(trial):

    params = {
        "n_estimators": 250,

        "max_depth": trial.suggest_int("max_depth", 3, 15),

        "min_samples_split": trial.suggest_int("min_samples_split", 2, 100),

        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),

        "n_jobs": -1,
        "random_state": 42
    }

    model = RandomForestRegressor(**params)

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

study2 = optuna.create_study(direction="maximize")

study2.optimize(
    objective,
    n_trials=15
)

print(study2.best_params)
print(-study2.best_value)

[I 2026-05-22 17:06:08,629] A new study created in memory with name: no-name-146994dc-eb70-4ec9-b860-6111fd13ada9
[I 2026-05-22 17:08:17,102] Trial 0 finished with value: -22.01278145849643 and parameters: {'max_depth': 14, 'min_samples_split': 12, 'min_samples_leaf': 15}. Best is trial 0 with value: -22.01278145849643.
[I 2026-05-22 17:08:50,104] Trial 1 finished with value: -23.584010023166154 and parameters: {'max_depth': 3, 'min_samples_split': 24, 'min_samples_leaf': 26}. Best is trial 0 with value: -22.01278145849643.
[I 2026-05-22 17:11:22,973] Trial 2 finished with value: -22.249077717146765 and parameters: {'max_depth': 14, 'min_samples_split': 93, 'min_samples_leaf': 31}. Best is trial 0 with value: -22.01278145849643.
[I 2026-05-22 17:13:33,384] Trial 3 finished with value: -22.315887791547862 and parameters: {'max_depth': 12, 'min_samples_split': 40, 'min_samples_leaf': 39}. Best is trial 0 with value: -22.01278145849643.
[I 2026-05-22 17:16:09,128] Trial 4 finished with va

{'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 2}
21.75975253001346


Los mejores modelos fueron:

1.

[I 2026-05-22 17:32:21,111]

Trial 13 finished with value: -21.75975253001346 and parameters: {'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 2}.


2.

[I 2026-05-22 17:08:17,102]

Trial 0 finished with value: -22.01278145849643 and parameters: {'max_depth': 14, 'min_samples_split': 12, 'min_samples_leaf': 15}


In [259]:
rfr_seg_opt = RandomForestRegressor(max_depth=15, min_samples_split=3, min_samples_leaf=2, n_jobs=-1, random_state=42)
rfr_seg_opt.fit(X_train, y_train)
scores = cross_val_score(rfr_seg_opt, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

y_pred_rfr_seg_opt = rfr_seg_opt.predict(X_test)
print('Modelo RFR segunda optimizacion - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_rfr_seg_opt):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_rfr_seg_opt):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_rfr_seg_opt):.2f}')

RMSE por fold: [21.72820873 21.79717041 21.81270068 21.58193231 21.6407037 ]
RMSE promedio: 21.71214316464755
Desvío estándar del RMSE: 0.08905954699576585
Modelo RFR segunda optimizacion - Test
Middle Square Error = 465.95
Root Middle Square Error = 21.59
R2 = 0.26


In [260]:
feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rfr_seg_opt.feature_importances_
})
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)
feature_importances

,Feature,Importance
46,2019,0.095927
3,loudness,0.083243
1,energy,0.069912
7,instrumentalness,0.066807
11,duration_ms,0.065282
0,danceability,0.061667
6,acousticness,0.059808
10,tempo,0.059807
5,speechiness,0.059749
9,valence,0.058815


#### Segunda submission

Correr esto:

In [287]:
X_train_reduced = X_train[vars]
X_test_reduced = X_test[vars]

rfr = RandomForestRegressor(n_jobs=-1, random_state=42)
rfr.fit(X_train_reduced, y_train)
scores = cross_val_score(rfr, X_train_reduced, y_train, cv=5, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

y_pred_rfr_base = rfr.predict(X_test_reduced)
print('Modelo RFR base - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_pred_rfr_base):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_pred_rfr_base):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_pred_rfr_base):.2f}')

RMSE por fold: [21.54614591 21.51291809 21.59737856 21.35177961 21.47796862]
RMSE promedio: 21.497238158220167
Desvío estándar del RMSE: 0.08269876853207118
Modelo RFR base - Test
Middle Square Error = 441.01
Root Middle Square Error = 21.00
R2 = 0.30


Hacemos el df_val

In [288]:
df_val = pd.read_csv('https://raw.githubusercontent.com/ichiP245/TP_Kaggle_Predictivo/refs/heads/main/base_val.csv')

In [289]:
def imputacion_anio(anio: int):
  if anio < 1980:
    return 1970
  if anio < 1990:
    return 1980
  if anio < 2000:
    return 1990
  if anio < 2010:
    return 2000
  if anio < 2015:
    return 2010
  return anio

In [290]:
df_val['anio'] = df_val['track_album_release_date'].apply(lambda x: x.split('-')[0])
df_val['anio'] = df_val['anio'].astype(int)

In [291]:
df_val['grupo_anio'] = df_val['anio'].apply(imputacion_anio)

In [292]:
df_val_orig = df_val.copy()

In [293]:
dummies_genre = pd.get_dummies(df_val['playlist_genre']).astype(int)
dummies_subgenre = pd.get_dummies(df_val['playlist_subgenre']).astype(int)
dummies_grupo_anio = pd.get_dummies(df_val['grupo_anio']).astype(int)
variables = ['danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'duration_ms']

df_val = pd.concat([df_val[variables], dummies_genre, dummies_subgenre, dummies_grupo_anio], axis=1)

In [294]:
df_val.columns = df_val.columns.astype(str)
df_val.columns = df_val.columns.astype(str)

In [295]:
df_val_rfr = df_val[list((set(df_val.columns) & set(X_train_reduced.columns)))]

In [300]:
df_val_rfr = df_val_rfr[list(X_train_reduced.columns)]

In [301]:
y_pred_val_rfr = rfr.predict(df_val_rfr)

In [302]:
submission_df_cb = pd.DataFrame({
    'ID': df_val_orig['Unnamed: 0'],
    'track_popularity': y_pred_val_rfr
})

submission_df_cb.to_csv('submission_rfr.csv', index=False)

### ExtraTreesRegressor

In [303]:
etr = ExtraTreesRegressor(n_estimators=500, random_state=42)
scores = cross_val_score(etr, X_train_reduced, y_train, cv=5, scoring='neg_root_mean_squared_error')
print("RMSE por fold:", -scores)
print("RMSE promedio:", -np.mean(scores))
print("Desvío estándar del RMSE:", np.std(-scores))

RMSE por fold: [21.89313187 21.70499452 21.94692418 21.60207591 21.73044464]
RMSE promedio: 21.775514222935506
Desvío estándar del RMSE: 0.1267283911951663


In [304]:
etr.fit(X_train_reduced, y_train)

ExtraTreesRegressor(n_estimators=500, random_state=42)

In [305]:
# Make predictions
y_etr = etr.predict(X_test_reduced)
print('Modelo ETR base - Test')
print(f'Middle Square Error = {mean_squared_error(y_test, y_etr):.2f}')
print(f'Root Middle Square Error = {root_mean_squared_error(y_test, y_etr):.2f}')   # Por cuanto erro, en unidades de y
print(f'R2 = {r2_score(y_test, y_etr):.2f}')

Modelo ETR base - Test
Middle Square Error = 454.71
Root Middle Square Error = 21.32
R2 = 0.28


### Fin